In [44]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn import svm as svm
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_validate, KFold
from sklearn.metrics import confusion_matrix as cm

## Loading data

In [45]:
def _to_array(raw_features):
    """
    Convert stored feature payload to a 3D array: (windows, channels, n_features).
    Handles the dict stored in features.npz.
    """
    arr = np.asarray(raw_features)
    if arr.dtype == object and arr.shape == ():  # dict wrapped in 0-d object
        obj = arr.item()
        if isinstance(obj, dict):
            keys = sorted(obj.keys())  # consistent order
            stacked = np.stack([np.asarray(obj[k]) for k in keys], axis=-1)
            return stacked, keys
        return np.asarray(obj), None
    return arr, None

In [46]:
def load_feature_files(root="data", pattern="*_features.npz", target_channels=23):
      root = Path(root)
      X_list, y_list, meta = [], [], []
      for fp in sorted(root.rglob(pattern)):
          data = np.load(fp, allow_pickle=True)
          X_raw = data["features"]
          X, feat_keys = _to_array(X_raw)
          channels = X.shape[1]
          if channels != target_channels:
              print(f"Skipping {fp} (channels={channels}, expected {target_channels})")
              continue
          labels = data.get("window_labels")
          labels = np.array([None] * len(X), dtype=object) if labels is None else np.asarray(labels, dtype=object)
          X_list.append(X)
          y_list.append(labels)
          meta.append({
              "path": str(fp),
              "feature_list": feat_keys or list(data["feature_list"]),
              "fs": float(np.asarray(data["fs"]).squeeze()),
              "window_size_samples": int(data["window_size_samples"]),
              "window_step_samples": int(data["window_step_samples"]),
              "window_start_samples": np.asarray(data["window_start_samples"]),
              "window_end_samples": np.asarray(data["window_end_samples"]),
              "window_label_confidence": data.get("window_label_confidence"),
          })
      if not X_list:
          raise FileNotFoundError(f"No feature files with {target_channels} channels found under {root}")
      return np.vstack(X_list), np.concatenate(y_list), meta

In [47]:
def load_features_by_label(root="data", pattern="*_features.npz", target_channels=23):
      X, y, meta = load_feature_files(root, pattern, target_channels=target_channels)
      mask = y != None  # noqa: E711
      return X[mask], y[mask], meta

### Classifiers, Lda

In [48]:
X_raw, y, meta = load_features_by_label(target_channels=23)
X = X_raw.reshape(X_raw.shape[0], -1)  # flatten channels×features

X_train, X_test, y_train, y_test = train_test_split(
      X, y, test_size=0.2, stratify=y, random_state=42
  )

lda = LDA(store_covariance=True)
lda.fit(X_train, y_train)
y_pred = lda.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Skipping data\practice data\emg_subjectS01_session01_features.npz (channels=4, expected 23)
Skipping data\practice data\emg_subjectS01_session02_features.npz (channels=4, expected 23)
Skipping data\practice data\emg_subjectS01_session03_features.npz (channels=4, expected 23)
Skipping data\practice data\emg_subjectS01_session04_features.npz (channels=4, expected 23)
Skipping data\practice data\emg_subjectS01_session05_features.npz (channels=4, expected 23)
Test accuracy: 0.75
              precision    recall  f1-score   support

   left_turn       0.86      0.71      0.78       389
     neutral       0.50      0.75      0.60       211
  right_turn       0.88      0.80      0.84       352

    accuracy                           0.75       952
   macro avg       0.75      0.75      0.74       952
weighted avg       0.79      0.75      0.76       952



In [ ]:
qda = QDA(store_covariance=True)
qda.fit(X_train, y_train)

qda_pred = qda.predict(X_test)
qda_pred_train = qda.predict(X_train)

print("test accuracy:", np.mean(qda_pred == L_test))
print("error rate:", np.mean(qda_pred != L_test))
print("train accuracy:", np.mean(qda_pred_train == L_train))

c:\Users\igdal\anaconda3\envs\capstone-emg\Lib\site-packages\sklearn\discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
c:\Users\igdal\anaconda3\envs\capstone-emg\Lib\site-packages\sklearn\discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
c:\Users\igdal\anaconda3\envs\capstone-emg\Lib\site-packages\sklearn\discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


array([[294,   7,  41],
       [ 85, 195, 140],
       [ 10,   9, 171]], dtype=int64)